In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.neural_network import MLPClassifier

# Read in the csv as a dataframe, getting rid of non-demographic data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Show how the data looks
df

In [ ]:
# Check if there is missing data
df['TotalCharges'] = pd.to_numeric(df.TotalCharges, errors='coerce')
df.isnull().sum()
# There is missing data in the TotalCharges column

In [ ]:
# Drop the customerID and TotalCharges fields
# As the customerID is pointless and the TotalCharges has missing data
df.drop(columns=['customerID', 'TotalCharges'], inplace=True)
df.info()

In [ ]:
# Show the value counts for all of the data
value_counts_dict = {col: df[col].value_counts() for col in df.columns}

for col, counts in value_counts_dict.items():
    print(f"\nValue counts for column: {col}")
    print(counts)

In [ ]:
if 'Churn' in df.columns:
    df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})
# Detect categorical columns (object or category type)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Apply One-Hot Encoding only to categorical columns
# Creates essentially boolean columns for the object values, but not the numeric
df = pd.get_dummies(df, columns=cat_cols)



In [ ]:
# Show information about the number of entries and types of data
# Many more columns are created from One-Hot Encoding
# Some of the 'No' categories and 'No x service' categories are brought
# together and encoded, but this makes sense as a column should be either
# True or False really, not have three options
df.info()

In [ ]:
# Again show the value counts, now that more columns have been created
for col in df.columns:
    print(f"\nColumn: {col}")
    value_counts = df[col].value_counts(dropna=False)
    print(value_counts)

In [ ]:
# Create the models
models = [
    LogisticRegression(class_weight='balanced', max_iter=1000),
    DecisionTreeClassifier(class_weight='balanced'),
    MLPClassifier(max_iter=1000)
]

for model in models:
  X = df.drop('Churn', axis=1)
  y = df['Churn']
  # Split the data into training and test sets
  # - test_size=0.2 is used to ensure that 20% of the data is used for testing
  # - random_state=1 : ensures the seed stays the same
  # - stratify=y: maintains the same distribution in churn
  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, random_state=1)
  model.fit(X_train, y_train)
  # Making the model more conservative in predicting churn greatly
  # increases the accuracy, 0.6 seems to be the sweet spot
  # in terms of accuracy, with it never predicting if the
  # value gets too high, but not being accurate if the
  # value is too low
  y_proba = model.predict_proba(X_test)[:, 1]
  y_pred = (y_proba > 0.6).astype(int)

  # Print the metrics
  accuracy = accuracy_score(y_test, y_pred)
  precision = precision_score(y_test, y_pred)
  recall = recall_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  confuse = confusion_matrix(y_test, y_pred)
  print(f'{model.__class__.__name__} Evaluation')
  print(f'Accurancy is {accuracy}')
  print(f'Precision is {precision}')
  print(f'Recall is {recall}')
  print(f'f1 is {f1}')
  print(f' Confusion Matrix is {confuse}')